# 🔵 Roundabouts in Metropolitan France — A Geospatial Data Analysis

> *"France is the land of roundabouts. Five are built every day and have been for thirty years."*  
> — Les Échos

---

**Author:** Perig Montfort  
**Data:** OpenStreetMap · INSEE · france-geojson

---

## Context & Motivation

With more than **60,000 roundabouts**, France is the world champion of the circular junction. This is no joke: the country alone accounts for more than half of all European roundabouts.

This project starts from a simple question: **do these roundabouts have a geography?** Are they larger in rural areas? More circular in cities? Are there "profiles" of roundabouts depending on the territory?

To answer these questions, we build a complete geospatial data mining pipeline:
1. **Data collection and analysis** — extraction from OpenStreetMap, geometric reconstruction, territorial enrichment
2. **Pattern Mining** — discovery of frequent patterns and association rules between roundabout features
3. **Clustering** — identification of roundabout profiles through unsupervised learning

---

## Notebook Structure

| Part | Content |
|------|---------|
| **Part 1** | Collection, reconstruction and exploratory data analysis |
| **Part 2** | Pattern Mining (Apriori + association rules) |
| **Part 3** | K-Means Clustering and territorial interpretation |

---

## ♻️ Reproducibility

This project is **almost entirely reproducible automatically** from this notebook.

**Only exception**: the INSEE grid must be downloaded manually.

👉 Download: https://www.insee.fr/fr/statistiques/2520034  
Dataset: **200m grid squares – Metropolitan France** in **GeoPackage (.gpkg)** format  
→ Place the file in `data/carreaux_200m_met.gpkg`

💡 Contact: perig.montfort@ens-lyon.fr


---

# 🗂️ PART 1 — Data Collection and Analysis

This first part covers all the preparatory work: installing dependencies, downloading and filtering OpenStreetMap data, geometric reconstruction of roundabouts, territorial enrichment and exploratory analysis.

---


## 1.1 — Installing Dependencies

The pipeline relies on several complementary tools:

| Tool | Role |
|------|------|
| `osmium-tool` | Ultra-fast filtering of OSM data (multi-GB PBF files) |
| `geopandas` / `shapely` | Manipulation, reconstruction and spatial geometry analysis |
| `scikit-learn` | Clustering (K-Means) and standardisation |
| `folium` | Generation of interactive HTML maps |
| `matplotlib` | Static visualisations |

The following cell automatically installs all required packages.


In [ ]:
import subprocess, sys

def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *packages])

print("📦 Installing Python packages...")
pip_install(
    "geopandas",
    "shapely",
    "pandas",
    "matplotlib",
    "folium",
    "numpy",
    "scikit-learn",
)
print("✅ Python packages installed.")


Installing `osmium-tool` — the OSM filtering tool. Several methods are tried sequentially (apt, conda, brew) to cover Linux, macOS and conda environments.


In [ ]:
import shutil, subprocess

def run(cmd, **kwargs):
    return subprocess.run(cmd, shell=isinstance(cmd, str),
                          capture_output=True, text=True, **kwargs)

if shutil.which("osmium") is None:
    print("🔧 osmium not found — attempting installation...")

    r = run("sudo apt-get install -y -q osmium-tool")
    if r.returncode == 0 and shutil.which("osmium"):
        print("✅ osmium installed via apt.")

    elif shutil.which("conda"):
        r2 = run("conda install -y -q -c conda-forge osmium-tool")
        if r2.returncode == 0:
            print("✅ osmium installed via conda.")
        else:
            print("❌ conda failed:\n", r2.stderr[-500:])

    elif shutil.which("brew"):
        r3 = run("brew install osmium-tool")
        if r3.returncode == 0:
            print("✅ osmium installed via brew.")
        else:
            print("❌ brew failed:\n", r3.stderr[-500:])

    else:
        print("❌ Unable to install osmium-tool automatically.")
        print("   Install manually: https://osmcode.org/osmium-tool/")
        raise SystemExit("osmium-tool required.")
else:
    print(f"✅ osmium already available: {shutil.which('osmium')}")


Defining file paths. Using `pathlib.Path` makes the notebook portable and OS-independent.


In [ ]:
from pathlib import Path

INPUT_PBF = Path("data/france_latest.osm.pbf")      # Full OSM extract (~5 GB)
FILTERED_PBF = Path("data/france_roundabouts.osm.pbf") # Filtered: junction=roundabout only
GEOJSON_WAYS = Path("data/roundabouts.geojson")        # GeoJSON export readable by GeoPandas
INPUT_CAR = Path("data/carreaux_200m_met.gpkg")     # INSEE 200m grid
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

print("📁 Output directory ready:", OUTPUT_DIR.resolve())


## 1.2 — Downloading and Filtering OSM Data

### Why OpenStreetMap?

OpenStreetMap (OSM) is a collaborative, freely available geographic database. For roundabouts, OSM uses the tag `junction=roundabout` on road segments forming a gyratory. It is the most comprehensive and up-to-date source available.

### Extraction Pipeline

```
france-latest.osm.pbf (~5 GB)
        │
        ▼  osmium tags-filter (junction=roundabout)
france_roundabouts.osm.pbf (~10 MB)
        │
        ▼  osmium export → GeoJSON
roundabouts.geojson
        │
        ▼  GeoPandas
GeoDataFrame (line segments)
```

⚠️ Downloading the France extract may take several minutes (~5 GB). Filtering steps are memoised: if intermediate files already exist, they are skipped.


In [ ]:
import requests

url = "http://download.openstreetmap.fr/extracts/europe/france-latest.osm.pbf"

if not INPUT_PBF.exists():
    print("⏳ Downloading OSM France file (~5 GB)...")
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(INPUT_PBF, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
    size_mb = INPUT_PBF.stat().st_size / 1e6
    print(f"✅ Download complete: {INPUT_PBF} ({size_mb:.1f} MB)")
else:
    size_mb = INPUT_PBF.stat().st_size / 1e6
    print(f"✅ File already present: {INPUT_PBF} ({size_mb:.1f} MB)")


**osmium filtering** — we extract only the arcs tagged `junction=roundabout`. `osmium tags-filter` is extremely fast even on multi-gigabyte files, as it reads the PBF sequentially without loading everything into memory.


In [ ]:
import time

if FILTERED_PBF.exists():
    size_mb = FILTERED_PBF.stat().st_size / 1e6
    print(f"✅ Filtering already done: {FILTERED_PBF} ({size_mb:.1f} MB)")
else:
    print("⏳ Running osmium tags-filter...")
    t0 = time.time()
    cmd_filter = [
        "osmium", "tags-filter",
        str(INPUT_PBF),
        "junction=roundabout",
        "-o", str(FILTERED_PBF),
        "--overwrite",
        "--progress",
    ]
    r = subprocess.run(cmd_filter)
    elapsed = time.time() - t0
    time.sleep(1)
    assert r.returncode == 0, f"❌ osmium tags-filter failed (code {r.returncode})"
    size_mb = FILTERED_PBF.stat().st_size / 1e6
    print(f"✅ Filtering done in {elapsed:.0f}s")
    print(f"   Filtered file: {FILTERED_PBF} ({size_mb:.1f} MB)")


**GeoJSON export** — converting the filtered PBF to GeoJSON, a standard format directly readable by GeoPandas. We export the `id` and `type` attributes to preserve OSM identifiers.


In [ ]:
if GEOJSON_WAYS.exists():
    size_mb = GEOJSON_WAYS.stat().st_size / 1e6
    print(f"✅ Export already done: {GEOJSON_WAYS} ({size_mb:.1f} MB)")
else:
    print("⏳ Exporting GeoJSON with osmium export...")
    t0 = time.time()
    cmd_export = [
        "osmium", "export",
        str(FILTERED_PBF),
        "--output-format", "geojson",
        "-o", str(GEOJSON_WAYS),
        "--overwrite",
        "--attributes", "id,type",
    ]
    r = subprocess.run(cmd_export, capture_output=True, text=True)
    elapsed = time.time() - t0
    time.sleep(0.1)
    assert r.returncode == 0, f"❌ osmium export failed (code {r.returncode})"
    size_mb = GEOJSON_WAYS.stat().st_size / 1e6
    print(f"✅ Export done in {elapsed:.1f}s — GeoJSON: {GEOJSON_WAYS} ({size_mb:.1f} MB)")


## 1.3 — Loading Data and Supplementary Sources

The OSM data provides the **raw geometry** of roundabouts (line segments). For territorial analysis, we enrich them with:

- **Administrative boundaries** (municipalities, departments, regions) — via [france-geojson](https://github.com/gregoiredavid/france-geojson)
- **INSEE 200m grid** — to classify each roundabout by its urbanisation level

> 💡 **Projection system**: we work in **EPSG:2154** (RGF93 / Lambert-93, the official French projection) for area and distance calculations, and in **EPSG:4326** (WGS84, latitude/longitude) for joins and visualisations.


In [ ]:
import geopandas as gpd
import warnings
warnings.filterwarnings("ignore")

# Raw OSM data
print(f"📂 Loading OSM data...")
t0 = time.time()
gdf = gpd.read_file(str(GEOJSON_WAYS))
print(f"   ✅ {len(gdf):,} segments loaded in {time.time()-t0:.1f}s")
print(f"   (These segments will be reconstructed as polygons in the next step)")

# INSEE grid
print("📂 Loading INSEE grid...")
gdf_carreaux = gpd.read_file("data/carreaux_200m_met.gpkg").to_crs("EPSG:2154")
gdf_carreaux["surface_km2"] = (gdf_carreaux.geometry.area / 1_000_000).round(4)
gdf_carreaux["densite"] = gdf_carreaux["ind"] / gdf_carreaux["surface_km2"]

def densite_to_urban(d):
    """Classifies a grid cell by its population density (inhabitants/km²)."""
    if d >= 1500: return "dense urban"
    elif d >= 300: return "urban"
    elif d >= 50: return "peri-urban"
    else: return "rural"

gdf_carreaux["urban_level"] = gdf_carreaux["densite"].apply(densite_to_urban)
print(f"   ✅ {len(gdf_carreaux):,} grid cells loaded")
print(f"   Distribution: {gdf_carreaux['urban_level'].value_counts().to_dict()}")

# Administrative boundaries
print("📂 Loading administrative boundaries...")
base_url = "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master"
gdf_communes = gpd.read_file(f"{base_url}/communes.geojson").to_crs("EPSG:4326")
gdf_departements = gpd.read_file(f"{base_url}/departements.geojson").to_crs("EPSG:4326")
gdf_regions = gpd.read_file(f"{base_url}/regions.geojson").to_crs("EPSG:4326")
print(f"   ✅ {len(gdf_communes):,} municipalities | {len(gdf_departements)} departments | {len(gdf_regions)} regions")

gdf.head(3)


## 1.4 — Geometric Reconstruction of Roundabouts

### The OSM Representation Problem

In OSM, a roundabout is **not** stored as a polygon. It is represented as a set of arcs (`LineString`) that, joined end-to-end, describe the circular outline of the road. Before any analysis, these polygons must therefore be **reconstructed**.

The procedure is:
1. Filter geometries of type `LineString` / `MultiLineString`
2. Merge all segments via `unary_union`
3. Extract closed polygons with `polygonize` (Shapely)

This step is critical: poor reconstruction leads to counting errors and problems during spatial joins.


In [ ]:
from shapely.ops import unary_union, polygonize
import numpy as np

# Filtering and reprojection to Lambert-93 (metric units)
df = (gdf[["@type", "@id", "highway", "geometry"]]
        .pipe(lambda d: d[d.geometry.geom_type.isin(["LineString", "MultiLineString"])])
        .set_crs("EPSG:4326", allow_override=True)
        .to_crs("EPSG:2154"))

print(f"   {len(df):,} LineString segments retained")

# Polygon reconstruction
print("🔄 Reconstructing roundabouts via polygonisation...")
t0 = time.time()
polygons = list(polygonize(unary_union(df.geometry)))
print(f"   ✅ {len(polygons):,} polygons reconstructed in {time.time()-t0:.1f}s")

gdf_poly = gpd.GeoDataFrame(
    {"roundabout_id": range(len(polygons))},
    geometry=polygons,
    crs="EPSG:2154"
)


### Removing Geometric Duplicates

After inspection, some polygons overlap — a consequence of duplicates in OSM or slightly different reconstructions for the same physical object. To avoid counting the same roundabout twice, we remove redundant geometries via a **spatial intersection join**, always keeping the polygon with the larger area (hypothesis: the more extensive geometry is the most faithful to reality).


In [ ]:
print("🧹 Detecting and removing geometric duplicates...")
t0 = time.time()

gdf_poly["_area_tmp"] = gdf_poly.geometry.area

overlaps = gpd.sjoin(
    gdf_poly[["roundabout_id", "_area_tmp", "geometry"]],
    gdf_poly[["roundabout_id", "_area_tmp", "geometry"]],
    how="inner", predicate="intersects"
)
overlaps = overlaps[overlaps["roundabout_id_left"] != overlaps["roundabout_id_right"]]

to_drop = set()
for _, row in overlaps.iterrows():
    l_id, r_id = row["roundabout_id_left"], row["roundabout_id_right"]
    l_area, r_area = row["_area_tmp_left"], row["_area_tmp_right"]
    if l_area < r_area:
        to_drop.add(l_id)
    elif r_area < l_area:
        to_drop.add(r_id)

n_before = len(gdf_poly)
gdf_poly = gdf_poly[~gdf_poly["roundabout_id"].isin(to_drop)].copy()
gdf_poly = gdf_poly.drop(columns="_area_tmp").reset_index(drop=True)
gdf_poly["roundabout_id"] = range(len(gdf_poly))

print(f"   {len(to_drop):,} duplicates removed in {time.time()-t0:.1f}s")
print(f"   {n_before:,} → {len(gdf_poly):,} roundabouts retained")


## 1.5 — Geometric Feature Engineering

From the reconstructed polygons, we compute several **geometric indicators** that will serve as variables for pattern mining and clustering.

| Variable | Formula | Interpretation |
|----------|---------|----------------|
| `area_m2` | Polygon area | Roundabout surface area |
| `perimeter_m` | Contour length | Road length |
| `diameter_m` | $2\sqrt{A/\pi}$ | Diameter of a circle with equivalent area |
| `circularity` | $4\pi A / P^2$ | 1 = perfect circle, 0 = flat shape |

The circularity index (also known as the **isoperimetric index**) is particularly useful: a poorly reconstructed or irregular roundabout will have a low circularity, which may indicate a data issue or an atypical geometry.


In [ ]:
gdf_poly["area_m2"]      = gdf_poly.geometry.area.round(2)
gdf_poly["perimeter_m"]  = gdf_poly.geometry.length.round(2)
gdf_poly["diameter_m"]   = (2 * np.sqrt(gdf_poly["area_m2"] / np.pi)).round(2)
gdf_poly["circularity"]  = (4 * np.pi * gdf_poly["area_m2"] / gdf_poly["perimeter_m"] ** 2).round(4)

centroids_wgs = gdf_poly.geometry.centroid.to_crs("EPSG:4326")
gdf_poly["lon"] = centroids_wgs.x.round(6)
gdf_poly["lat"] = centroids_wgs.y.round(6)

print("✅ Features géométriques calculées :")
gdf_poly[["area_m2", "perimeter_m", "diameter_m", "circularity"]].describe().round(2)

## 1.6 — Territorial Enrichment via Spatial Joins

We attach to each roundabout its **territorial context** via spatial joins (point-in-polygon):
- Municipality, department, region — via administrative boundaries
- Urbanisation level — via the INSEE grid

For roundabouts falling outside a polygon (border areas, boundary artefacts), we use `sjoin_nearest` as a fallback: we assign the nearest territory.


In [ ]:
gdf_pts = gdf_poly.to_crs("EPSG:4326").copy()
gdf_pts["geometry"] = gdf_pts.geometry.representative_point()

def spatial_join(gdf_pts, gdf_ref):
    """Spatial join with nearest fallback for points outside polygons."""
    cols = ["roundabout_id", "geometry"]
    join = gpd.sjoin(
        gdf_pts[cols], gdf_ref[["nom", "geometry"]],
        how="left", predicate="within"
    ).drop_duplicates("roundabout_id").set_index("roundabout_id")

    missing = gdf_pts[gdf_pts["roundabout_id"].isin(join[join["nom"].isna()].index)]
    if not missing.empty:
        join_nearest = gpd.sjoin_nearest(
            missing[cols], gdf_ref[["nom", "geometry"]], how="left"
        ).drop_duplicates("roundabout_id").set_index("roundabout_id")
        join["nom"] = join["nom"].combine_first(join_nearest["nom"])

    return join["nom"].reindex(gdf_poly["roundabout_id"]).values

print("🗺️  Running spatial joins...")
for gdf_ref, col in [(gdf_communes, "commune"), (gdf_departements, "departement"), (gdf_regions, "region")]:
    gdf_poly[col] = spatial_join(gdf_pts, gdf_ref)
    print(f"   ✅ {col} — {gdf_poly[col].notna().sum():,} roundabouts matched")

# Urbanisation level via INSEE grid
gdf_pts_m = gdf_poly[["roundabout_id", "geometry"]].copy()
gdf_pts_m["geometry"] = gdf_pts_m.geometry.centroid

def sjoin_urban(pts, method="within"):
    fn = gpd.sjoin_nearest if method == "nearest" else gpd.sjoin
    kwargs = {} if method == "nearest" else {"predicate": "within"}
    return fn(pts, gdf_carreaux[["urban_level", "geometry"]], how="left", **kwargs)

join = sjoin_urban(gdf_pts_m, method="within")
missing_pts = gdf_pts_m[gdf_pts_m["roundabout_id"].isin(join[join["urban_level"].isna()]["roundabout_id"])]
join_nearest = sjoin_urban(missing_pts, method="nearest")

gdf_poly["urban_level"] = (
    join.drop_duplicates("roundabout_id").set_index("roundabout_id")["urban_level"]
    .combine_first(join_nearest.drop_duplicates("roundabout_id").set_index("roundabout_id")["urban_level"])
    .reindex(gdf_poly["roundabout_id"]).values
)
print(f"   ✅ urban_level — distribution: {gdf_poly['urban_level'].value_counts().to_dict()}")

print(f"\n🎉 Final dataset: {len(gdf_poly):,} enriched roundabouts")
gdf_poly.head(3)


## 1.7 — Exploratory Analysis

### Descriptive Statistics

Before any modelling, it is essential to explore the distribution of variables. We focus in particular on diameters (the central variable) and the breakdown by urbanisation level.


In [ ]:
import pandas as pd

print("=" * 50)
print("📊 GENERAL STATISTICS")
print("=" * 50)
print(f"  Total number of roundabouts: {len(gdf_poly):,}")
print()
print("  Diameter (m):")
print(f"    Mean   : {gdf_poly['diameter_m'].mean():.1f} m")
print(f"    Median : {gdf_poly['diameter_m'].median():.1f} m")
print(f"    Std    : {gdf_poly['diameter_m'].std():.1f} m")
print(f"    Min/Max: {gdf_poly['diameter_m'].min():.1f} m / {gdf_poly['diameter_m'].max():.1f} m")
print()
print("  Circularity:")
print(f"    Mean   : {gdf_poly['circularity'].mean():.3f}")
print(f"    Median : {gdf_poly['circularity'].median():.3f}")
print()
print("  Breakdown by urbanisation level:")
vc = gdf_poly["urban_level"].value_counts()
for level, count in vc.items():
    pct = 100 * count / len(gdf_poly)
    print(f"    {level:20s} : {count:7,} ({pct:.1f}%)")
print()
print("  Top 5 regions by number of roundabouts:")
print(gdf_poly["region"].value_counts().head())


### Static Visualisations

We display the **spatial density of roundabouts** and **population density** on two side-by-side hexbin maps, enabling a direct visual comparison between the distribution of roundabouts and that of the population.

Bar charts complement this view with an analysis by municipality and by region.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

xmin, ymin, xmax, ymax = gdf_regions.total_bounds

cities = {
    "Paris": (48.8566, 2.3522), "Lyon": (45.7640, 4.8357),
    "Marseille": (43.2965, 5.3698), "Toulouse": (43.6047, 1.4442),
    "Bordeaux": (44.8378, -0.5792), "Lille": (50.6292, 3.0573),
    "Nantes": (47.2184, -1.5536), "Strasbourg": (48.5734, 7.7521),
    "Nice": (43.7102, 7.2620), "Montpellier": (43.6119, 3.8772),
}

def add_cities(ax, col="white"):
    for name, (lat, lon) in cities.items():
        ax.scatter(lon, lat, s=18, color="cyan", edgecolor="black", zorder=10)
        ax.text(lon + 0.1, lat + 0.08, name, fontsize=7, color=col,
                fontweight="bold", zorder=11)

def add_borders(ax):
    gdf_regions.boundary.plot(ax=ax, color="white", linewidth=0.5, alpha=0.75)
    gdf_regions.dissolve().boundary.plot(ax=ax, color="black", linewidth=1.0)

def clean_ax(ax):
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel(""); ax.set_ylabel("")
    ax.set_frame_on(False)
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

fig = plt.figure(figsize=(22, 15))
fig.suptitle("Roundabout Analysis in Metropolitan France",
             fontsize=18, fontweight="bold", y=0.98)
fig.set_facecolor("#f7f0ff")

gs = fig.add_gridspec(1, 2, width_ratios=[1.1, 1], wspace=0.35)
gs_left  = gs[0].subgridspec(2, 1, hspace=0.3)
gs_right = gs[1].subgridspec(3, 1, hspace=0.55)

ax_map1 = fig.add_subplot(gs_left[0])
ax_map2 = fig.add_subplot(gs_left[1])
ax_hist = fig.add_subplot(gs_right[0])
ax_com = fig.add_subplot(gs_right[1])
ax_reg = fig.add_subplot(gs_right[2])

# --- Map 1: roundabout density ---
x_rp = gdf_poly["lon"].values
y_rp = gdf_poly["lat"].values
gdf_regions.dissolve().plot(ax=ax_map1, color="#1a0a2e", zorder=0)
hb1 = ax_map1.hexbin(x_rp, y_rp, gridsize=120, cmap="magma", bins="log",
                      extent=[xmin, xmax, ymin, ymax])
add_cities(ax_map1, "white"); add_borders(ax_map1); clean_ax(ax_map1)
plt.colorbar(hb1, ax=ax_map1, fraction=0.04, pad=0.02).set_label("Density (log)")
ax_map1.set_title(f"Spatial density of roundabouts ({len(gdf_poly):,})", fontsize=12)

# --- Map 2: population density (INSEE grid) ---
gdf_carreaux_wgs = gdf_carreaux.to_crs("EPSG:4326")
centroids_c = gdf_carreaux_wgs.geometry.centroid
x_urb = centroids_c.x.values
y_urb = centroids_c.y.values
weights = gdf_carreaux_wgs["ind"].fillna(0).clip(lower=0.01).values

gdf_regions.dissolve().plot(ax=ax_map2, color="#1a0a2e", zorder=0)
hb2 = ax_map2.hexbin(x_urb, y_urb, C=weights, reduce_C_function=np.sum,
                      gridsize=120, cmap="magma", bins="log",
                      extent=[xmin, xmax, ymin, ymax])
add_cities(ax_map2, "black"); add_borders(ax_map2); clean_ax(ax_map2)
plt.colorbar(hb2, ax=ax_map2, fraction=0.04, pad=0.02).set_label("Population (log)")
ax_map2.set_title("Population density (INSEE grid)", fontsize=12)

# --- Diameter distribution ---
diameters = gdf_poly["diameter_m"].clip(upper=100)
ax_hist.hist(diameters, bins=60, color="#8a4fff", edgecolor="white", linewidth=0.3)
ax_hist.axvline(diameters.mean(), color="red", linestyle="--",
                label=f"Mean: {diameters.mean():.1f} m")
ax_hist.axvline(diameters.median(), color="orange", linestyle="--",
                label=f"Median: {diameters.median():.1f} m")
ax_hist.set_xlabel("Diameter (m)")
ax_hist.set_ylabel("Number of roundabouts")
ax_hist.set_title("Diameter distribution (truncated at 100m)")
ax_hist.legend(fontsize=9)
ax_hist.grid(alpha=0.3)

# --- Top municipalities ---
top_com = gdf_poly["commune"].value_counts().head(10)
ax_com.barh(top_com.index[::-1], top_com.values[::-1], color="#8a4fff")
ax_com.set_xlabel("Number of roundabouts")
ax_com.set_title("Top 10 municipalities")
ax_com.grid(alpha=0.3, axis="x")

# --- By region ---
top_reg = gdf_poly["region"].value_counts()
ax_reg.barh(top_reg.index[::-1], top_reg.values[::-1], color="#5b8fff")
ax_reg.set_xlabel("Number of roundabouts")
ax_reg.set_title("Roundabouts by region")
ax_reg.grid(alpha=0.3, axis="x")

fig_path = OUTPUT_DIR / "roundabouts_overview.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Figure saved: {fig_path}")

### Interactive Map

An interactive Folium map allows individual roundabouts to be explored, geometric reconstruction consistency to be verified, and potential local anomalies to be detected. Each marker displays the roundabout's attributes (diameter, municipality, urban level…).

⚠️ Due to the number of objects, the map may be slow to display in the notebook — a standalone HTML version is saved in `output/`.


In [ ]:
import folium
from folium.plugins import MarkerCluster

gdf_poly_wgs = gdf_poly.to_crs("EPSG:4326").copy()

m = folium.Map(
    location=[gdf_poly_wgs["lat"].mean(), gdf_poly_wgs["lon"].mean()],
    zoom_start=6,
    tiles="CartoDB positron"
)
cluster = MarkerCluster(name="Roundabouts").add_to(m)

for _, row in gdf_poly_wgs.iterrows():
    commune = row.get("commune", "N/A") if pd.notna(row.get("commune")) else "N/A"
    departement = row.get("departement", "N/A") if pd.notna(row.get("departement")) else "N/A"
    region = row.get("region", "N/A") if pd.notna(row.get("region")) else "N/A"

    popup_html = (
        f"<b>ID</b>: {row['roundabout_id']}<br>"
        f"<b>Diameter</b>: {row['diameter_m']:.1f} m<br>"
        f"<b>Area</b>: {row['area_m2']:.0f} m²<br>"
        f"<b>Circularity</b>: {row['circularity']:.3f}<br>"
        f"<b>Municipality</b>: {commune}<br>"
        f"<b>Department</b>: {departement}<br>"
        f"<b>Region</b>: {region}<br>"
        f"<b>Urban level</b>: {row['urban_level']}"
    )
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=3,
        color="rebeccapurple",
        fill=True, fill_color="mediumorchid", fill_opacity=0.7,
        weight=1,
        popup=folium.Popup(popup_html, max_width=260)
    ).add_to(cluster)

folium.LayerControl().add_to(m)

html_map = OUTPUT_DIR / "roundabouts_map.html"
m.save(str(html_map))
print(f"✅ Interactive map saved: {html_map}")

try:
    from IPython.display import display
    display(m)
except Exception:
    pass

### Part 1 Summary

At the end of this first part, we have a clean, enriched dataset of **~65,000 roundabouts** (the exact count depends on the OSM extraction date), described by:
- **geometric attributes** (diameter, circularity, area)
- **administrative context** (municipality, department, region)
- **urbanisation level** derived from the INSEE grid

The exploratory analysis confirms intuitive patterns: roundabouts are more numerous in peri-urban areas and mid-sized towns. The diameter distribution is strongly skewed (median ~20m, with a few outliers >100m).

These observations serve as a starting point for the next two parts.


---

# 🔍 PART 2 — Pattern Mining

Here we aim to go beyond descriptive statistics: **are there frequent, non-trivial associations between roundabout features?**

For example:
- Are *small* roundabouts systematically associated with *dense urban* areas?
- Do certain regions have particular geometric profiles?

To answer these questions, we use **transactional pattern mining**, a classical knowledge extraction technique. The principle: represent each roundabout as a **set of items** (its categorical features) and search for combinations of items that frequently appear together.

---


## 2.1 — Preparation: Discretisation and Transactional Encoding

### Why Discretise?

The Apriori algorithm works with **categorical variables** (presence/absence of items). Continuous variables such as `diameter_m` must therefore be **discretised** into classes before being included in transactions.

We define 4 diameter classes based on the observed distribution:

| Class | Diameter | Interpretation |
|-------|----------|----------------|
| `very small` | < 10m | Very small roundabouts (dense areas) |
| `small` | 10–20m | The majority of urban roundabouts |
| `medium` | 20–40m | Standard roundabouts outside built-up areas |
| `large` | > 40m | Large-traffic roundabouts, rural areas |

### Transactional Format

Each roundabout becomes a **transaction**: a set of items such as `{small, dense urban, Île-de-France}`. These transactions are then encoded as a binary matrix (presence/absence).


In [ ]:
import pandas as pd

df = gdf_poly.copy()

df["diameter_cat"] = pd.cut(
    df["diameter_m"],
    bins=[0, 10, 20, 40, 100],
    labels=["very small", "small", "medium", "large"]
)

df_patterns = df[["diameter_cat", "urban_level", "region"]].dropna()

print(f"✅ {len(df_patterns):,} roundabouts retained for pattern mining")
print("\nDiameter class distribution:")
print(df["diameter_cat"].value_counts().sort_index())

s = df_patterns.stack()
df_encoded = (pd.crosstab(s.index.get_level_values(0), s) > 0)

print(f"\n📋 Transactional matrix: {df_encoded.shape[0]:,} transactions × {df_encoded.shape[1]} items")
df_encoded.head(3)

## 2.2 — Apriori Algorithm (from-scratch implementation)

### Theoretical Background

The **Apriori** algorithm (Agrawal & Srikant, 1994) is the founding algorithm of pattern mining. It relies on the **anti-monotonicity** property: if an itemset is frequent, all its subsets are also frequent (and conversely, if a subset is not frequent, the full itemset cannot be frequent either).

**Algorithm steps:**
1. Compute the support of all individual items → keep those ≥ `min_support`
2. Generate candidates of size $k+1$ by union of frequent itemsets of size $k$
3. Prune candidates whose subset is not frequent (Apriori property)
4. Compute the support of remaining candidates → keep the frequent ones
5. Repeat until no new frequent itemset is found

**Support** of an itemset $I$: $\text{support}(I) = \frac{\text{nb transactions containing } I}{\text{total nb transactions}}$


In [ ]:
def apriori(df_encoded, min_support=0.01):
    """Apriori algorithm
    Parameters:
    df_encoded: boolean DataFrame (transactions x items)
    min_support: minimum support threshold (between 0 and 1)
    
    Returns:
    DataFrame with columns 'itemsets' (frozenset) and 'support'
    """

    def generate_candidates(previous_itemsets, k):
        """Generates candidates of size k by union of itemsets of size k-1.
        Applies Apriori pruning: a candidate is kept only if
        all its subsets of size k-1 are frequent.
        """
        C = set()
        previous_list = list(previous_itemsets)

        for i in range(len(previous_list)):
            for j in range(i + 1, len(previous_list)):
                union = previous_list[i] | previous_list[j]
                if len(union) == k:
                    items_list = list(union)
                    # Apriori check: all (k-1) subsets must be frequent
                    if all(
                        frozenset(items_list[:l] + items_list[l+1:]) in previous_itemsets
                        for l in range(len(items_list))
                    ):
                        C.add(union)
        return C

    results = []

    # Step 1: frequent items of size 1
    supports_1 = df_encoded.mean()
    F = {frozenset([col]): sup for col, sup in supports_1.items() if sup >= min_support}
    print(f"   Frequent items (k=1): {len(F)}")

    k = 2
    while F:
        for itemset, support in F.items():
            results.append({"support": support, "itemsets": itemset})

        C = generate_candidates(set(F.keys()), k)

        new_frequents = {}
        for candidate in C:
            support = df_encoded[list(candidate)].min(axis=1).mean()
            if support >= min_support:
                new_frequents[candidate] = support

        print(f"   Frequent items (k={k}): {len(new_frequents)} (candidates tested: {len(C)})")
        F = new_frequents
        k += 1

    frequent_itemsets = pd.DataFrame(results)
    if not frequent_itemsets.empty:
        frequent_itemsets = frequent_itemsets.sort_values("support", ascending=False).reset_index(drop=True)

    return frequent_itemsets

In [ ]:
print(f"⚙️  Running Apriori (min_support=0.01)...")
t0 = time.time()
frequent_itemsets = apriori(df_encoded, min_support=0.01)
print(f"\n✅ {len(frequent_itemsets):,} frequent itemsets extracted in {time.time()-t0:.1f}s")
print()
print("Top 10 itemsets by support:")
frequent_itemsets.head(10)

## 2.3 — Association Rules

From the frequent itemsets, we generate **association rules** of the form:

$$\text{antecedent} \Rightarrow \text{consequent}$$

Each rule is evaluated according to three metrics:

| Metric | Definition | Interpretation |
|--------|-----------|----------------|
| **Support** | $P(A \cup B)$ | Frequency of the rule in the dataset |
| **Confidence** | $P(B \mid A)$ | Probability of B given A |
| **Lift** | $P(B\mid A) / P(B)$ | Enrichment factor relative to independence — lift > 1 indicates a positive association |

We filter here on `lift ≥ 1.2` to retain only statistically meaningful associations.


In [ ]:
def association_rules(frequent_itemsets, min_lift=1.2):
    """Generates association rules from frequent itemsets.
    
    Retains only rules with lift >= min_lift.
    """
    support_dict = dict(zip(frequent_itemsets["itemsets"], frequent_itemsets["support"]))
    rules = []

    for itemset in frequent_itemsets["itemsets"]:
        if len(itemset) < 2:
            continue

        items = list(itemset)
        n = len(items)

        for i in range(1, 2**n - 1):
            antecedent = frozenset([items[j] for j in range(n) if (i >> j) & 1])
            consequent = itemset - antecedent
            if not consequent:
                continue
            if antecedent not in support_dict or consequent not in support_dict:
                continue

            sup_ab = support_dict[itemset]
            sup_a = support_dict[antecedent]
            sup_b = support_dict[consequent]

            confidence = sup_ab / sup_a
            lift = confidence / sup_b

            if lift >= min_lift:
                rules.append({
                    "antecedents": antecedent,
                    "consequents": consequent,
                    "support": sup_ab,
                    "confidence": confidence,
                    "lift": lift
                })

    rules = pd.DataFrame(rules)
    if not rules.empty:
        rules = rules.sort_values("lift", ascending=False).reset_index(drop=True)
    return rules

rules = association_rules(frequent_itemsets, min_lift=1.2)
print(f"✅ {len(rules):,} rules extracted (lift ≥ 1.2)")
rules.head(10)

In [ ]:
# Cleanup and confidence filter
rules_clean = rules.copy()
rules_clean["antecedents"] = rules_clean["antecedents"].apply(lambda x: ", ".join(sorted(list(x))))
rules_clean["consequents"] = rules_clean["consequents"].apply(lambda x: ", ".join(sorted(list(x))))
rules_clean = (
    rules_clean[rules_clean["confidence"] >= 0.2]
    .sort_values(by=["lift", "confidence"], ascending=False)
    .reset_index(drop=True)
    .round(4)
)

print(f"📋 {len(rules_clean)} rules after filter (confidence ≥ 0.2)")
rules_clean

## 2.4 — Visualising Association Rules


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Association Rules Analysis", fontsize=14, fontweight="bold")

# Scatter: support vs confidence coloured by lift
sc = axes[0].scatter(
    rules_clean["support"],
    rules_clean["confidence"],
    c=rules_clean["lift"],
    cmap="plasma",
    s=40, alpha=0.8, edgecolors="none"
)
plt.colorbar(sc, ax=axes[0], label="Lift")
axes[0].set_xlabel("Support")
axes[0].set_ylabel("Confidence")
axes[0].set_title("Support × Confidence (colour = Lift)")
axes[0].grid(alpha=0.3)

# Top 15 rules by lift
top_rules = rules_clean.head(15)
labels = top_rules["antecedents"] + " → " + top_rules["consequents"]
axes[1].barh(labels[::-1], top_rules["lift"][::-1], color="#8a4fff")
axes[1].axvline(1.0, color="red", linestyle="--", label="Lift = 1 (independence)")
axes[1].set_xlabel("Lift")
axes[1].set_title("Top 15 rules by lift")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3, axis="x")

plt.tight_layout()
fig_path = OUTPUT_DIR / "pattern_mining_rules.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Figure saved: {fig_path}")

## 2.5 — Interpretation of Results

The association rules reveal several interesting structures in the data.

**Size and urbanisation:** The strongest and most robust relationship is between roundabout size and urban environment. Small roundabouts (< 20m) are strongly associated with dense urban areas — consistent with land constraints in urban settings. Conversely, large roundabouts (> 40m) are found mainly in peri-urban and rural areas, where available space allows it and where through-traffic (national roads, interchanges) requires larger gyratories.

**Regional specificities:** Certain regions reinforce these trends. Île-de-France, Nouvelle-Aquitaine and Occitanie generate significant regional associations — particularly for small urban roundabouts. Occitanie, which holds the record for the number of roundabouts, displays varied patterns reflecting the diversity of its territory (from Toulouse to Lozère).

**Absent patterns:** Bourgogne-Franche-Comté does not appear in the significant rules — a sign not of missing data, but of a distribution too homogeneous or too scattered to produce dominant associations. Corsica, on the other hand, is absent due to volume: few roundabouts = low support.

These results confirm that the geography of roundabouts is not random — it reflects spatial planning logics clearly structured by urban density.


---

# 🧩 PART 3 — Clustering

Pattern mining has highlighted associations between features. But these associations are binary and categorical. We now want to go further: **are there natural groups of roundabouts with coherent geometric profiles?**

K-Means clustering will allow us to answer this question in an unsupervised way, partitioning roundabouts by geometric similarity — without using urbanisation level as an input variable, so that it can then be compared to the clusters obtained.

---


## 3.1 — Building Features and Standardisation

### Variable Selection

We retain three variables for clustering:

| Variable | Description | Why? |
|----------|-------------|------|
| `diameter_m` | Roundabout diameter | Captures size, strongly linked to territorial context |
| `circularity` | Isoperimetric index | Captures shape — geometric anomalies vs standard roundabouts |
| `local_density` | Mean distance to the 10 nearest neighbours | Captures spatial context — isolated vs clustered roundabouts |

We do **not** include `urban_level` in the features: this will allow us to validate the clustering by cross-referencing the groups obtained with this variable.

### Standardisation

K-Means is sensitive to scale differences between variables (a diameter in metres vs an index between 0 and 1). We therefore standardise all variables using a `StandardScaler` (mean 0, standard deviation 1).


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

df_cluster = gdf_poly[["diameter_m", "circularity", "lat", "lon", "urban_level"]].dropna().copy()
print(f"   {len(df_cluster):,} roundabouts retained for clustering")

# Local density: mean distance to the 10 nearest neighbours (in lat/lon degrees)
coords = df_cluster[["lat", "lon"]].values
nbrs = NearestNeighbors(n_neighbors=10).fit(coords)
distances, _ = nbrs.kneighbors(coords)
df_cluster["local_density"] = distances.mean(axis=1)

print(f"   local_density — mean: {df_cluster['local_density'].mean():.4f}°, "
      f"median: {df_cluster['local_density'].median():.4f}°")

# Features and standardisation
X = df_cluster[["diameter_m", "circularity", "local_density"]]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("\n✅ Standardised features:")
print(pd.DataFrame(X_scaled, columns=X.columns).describe().round(3))

## 3.2 — Choosing the Number of Clusters: Elbow Method

The number of clusters $k$ is a hyperparameter that must be set upfront. The **elbow method** consists of plotting the **within-cluster inertia** (sum of squared distances from each point to its centroid) as a function of $k$.

Inertia always decreases as $k$ increases, but the gain decreases too. We look for the "elbow" of the curve — the point beyond which increasing $k$ no longer brings significant gain.


In [ ]:
from sklearn.cluster import KMeans

inertias = []
k_values = range(1, 9)

print("⚙️  Computing inertia for k=1 to 8...")
for k in k_values:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    print(f"   k={k} — inertie : {km.inertia_:,.0f}")

# Compute marginal gain to help read the elbow
gains = [inertias[i-1] - inertias[i] for i in range(1, len(inertias))]
gains_rel = [g / inertias[i] * 100 for i, g in enumerate(gains, start=1)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Elbow Method — Choosing the Number of Clusters", fontsize=13)

ax1.plot(k_values, inertias, marker="o", color="#8a4fff", linewidth=2)
ax1.fill_between(k_values, inertias, alpha=0.1, color="#8a4fff")
ax1.set_xlabel("Number of clusters k")
ax1.set_ylabel("Within-cluster inertia")
ax1.set_title("Inertia")
ax1.grid(alpha=0.3)

ax2.bar(range(2, len(k_values)+1), gains_rel, color="#5b8fff", edgecolor="white")
ax2.set_xlabel("k")
ax2.set_ylabel("Relative gain (%)")
ax2.set_title("Marginal inertia gain (%)")
ax2.grid(alpha=0.3, axis="y")

plt.tight_layout()
fig_path = OUTPUT_DIR / "clustering_elbow.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Figure saved: {fig_path}")

## 3.3 — Applying K-Means

We choose **k=4**: the elbow of the inertia curve is at this point, and this number allows a concrete interpretation of the profiles.


In [ ]:
k = 4

kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
df_cluster["cluster"] = kmeans.fit_predict(X_scaled)

print("✅ K-Means applied (k=4)")
print("\nCluster sizes:")
vc = df_cluster["cluster"].value_counts().sort_index()
for cluster_id, count in vc.items():
    pct = 100 * count / len(df_cluster)
    print(f"   Cluster {cluster_id} : {count:7,} ({pct:.1f}%)")

## 3.4 — Describing and Interpreting the Clusters


In [ ]:
# Statistics by cluster
cluster_summary = df_cluster.groupby("cluster")[["diameter_m", "circularity", "local_density"]].agg(
    ["mean", "median", "std"]
).round(2)
print("📊 Statistics by cluster:")
cluster_summary

In [ ]:
# Comparison with urbanisation level
urban_by_cluster = (
    df_cluster.groupby("cluster")["urban_level"]
    .value_counts(normalize=True)
    .rename("proportion")
    .mul(100).round(1)
    .unstack(fill_value=0)
)
print("📊 Urbanisation level composition by cluster (%):")
urban_by_cluster

### Cluster Profiles

The analysis of means and urban composition identifies four distinct profiles:

| Cluster | Mean diameter | Local density | Circularity | Profile |
|---------|--------------|---------------|-------------|--------|
| **0** | ~25m | Moderate | High | Intermediate urban/peri-urban roundabouts — transition zones |
| **1** | ~21m | Low (dense) | High | **Dense urban roundabouts** — dominant profile, small size, high concentration |
| **2** | ~50m | High (sparse) | High | **Large peri-urban/rural roundabouts** — transit routes, available space |
| **3** | ~22m | Variable | **Low** | **Atypical cases** — irregular shapes, imperfect geometric reconstructions |

**Convergence with pattern mining:** Clusters 1 and 2 exactly confirm the association rules extracted in Part 2: small roundabouts ↔ dense areas, large roundabouts ↔ low-density areas. Clustering additionally provides an intermediate profile (cluster 0) and a geometric outlier group (cluster 3), invisible in the association rules.


## 3.5 — Cluster Visualisations


In [ ]:
from mpl_toolkits.mplot3d import Axes3D

cluster_colors = ["#5b8fff", "#ff6b6b", "#2ecc71", "#f39c12"]
cluster_labels = {
    0: "Cl. 0 — Intermediate",
    1: "Cl. 1 — Dense urban",
    2: "Cl. 2 — Peri-urban/rural",
    3: "Cl. 3 — Atypical"
}

fig = plt.figure(figsize=(18, 14))
fig.suptitle("K-Means Cluster Visualisation (k=4)", fontsize=15, fontweight="bold")

# --- 3D View ---
ax1 = fig.add_subplot(2, 2, 1, projection="3d")
for cid, color in enumerate(cluster_colors):
    mask = df_cluster["cluster"] == cid
    ax1.scatter(
        df_cluster.loc[mask, "diameter_m"],
        df_cluster.loc[mask, "local_density"],
        df_cluster.loc[mask, "circularity"],
        c=color, label=cluster_labels[cid], alpha=0.3, s=8
    )
ax1.set_xlabel("Diameter (m)"); ax1.set_ylabel("Local density"); ax1.set_zlabel("Circularity")
ax1.set_title("3D View")
ax1.legend(fontsize=7, loc="upper right")

# --- Diameter × Local density ---
ax2 = fig.add_subplot(2, 2, 2)
for cid, color in enumerate(cluster_colors):
    mask = df_cluster["cluster"] == cid
    ax2.scatter(df_cluster.loc[mask, "diameter_m"], df_cluster.loc[mask, "local_density"],
                c=color, label=cluster_labels[cid], alpha=0.3, s=8)
ax2.set_xlabel("Diameter (m)"); ax2.set_ylabel("Local density")
ax2.set_title("Diameter × Local density")
ax2.legend(fontsize=7); ax2.grid(alpha=0.3)

# --- Diameter × Circularity ---
ax3 = fig.add_subplot(2, 2, 3)
for cid, color in enumerate(cluster_colors):
    mask = df_cluster["cluster"] == cid
    ax3.scatter(df_cluster.loc[mask, "diameter_m"], df_cluster.loc[mask, "circularity"],
                c=color, label=cluster_labels[cid], alpha=0.3, s=8)
ax3.set_xlabel("Diameter (m)"); ax3.set_ylabel("Circularity")
ax3.set_title("Diameter × Circularity")
ax3.legend(fontsize=7); ax3.grid(alpha=0.3)

# --- Heatmap: cluster × urbanisation ---
ax4 = fig.add_subplot(2, 2, 4)
im = ax4.imshow(urban_by_cluster.values, cmap="YlOrRd", aspect="auto")
plt.colorbar(im, ax=ax4, label="%")
ax4.set_xticks(range(len(urban_by_cluster.columns)))
ax4.set_xticklabels(urban_by_cluster.columns, rotation=30, ha="right", fontsize=9)
ax4.set_yticks(range(len(urban_by_cluster.index)))
ax4.set_yticklabels([f"Cluster {i}" for i in urban_by_cluster.index])
ax4.set_title("Composition by level of urbanization (%)")
for i in range(len(urban_by_cluster.index)):
    for j in range(len(urban_by_cluster.columns)):
        ax4.text(j, i, f"{urban_by_cluster.values[i, j]:.0f}",
                 ha="center", va="center", fontsize=9, fontweight="bold")

plt.tight_layout()
fig_path = OUTPUT_DIR / "clustering_results.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Figure saved: {fig_path}")

## 3.6 — Interactive Cluster Map


In [ ]:
import folium

cluster_colors_map = {0: "blue", 1: "red", 2: "green", 3: "orange"}

m = folium.Map(location=[46.6, 2.5], zoom_start=6, tiles="CartoDB positron")

# One layer per cluster
cluster_layers = {
    cid: folium.FeatureGroup(name=f"{cluster_labels[cid]}", show=True)
    for cid in sorted(df_cluster["cluster"].unique())
}

for _, row in df_cluster.iterrows():
    cid = row["cluster"]
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=2,
        color=cluster_colors_map.get(cid, "black"),
        fill=True, fill_opacity=0.6,
        tooltip=f"{cluster_labels[cid]} | {row['urban_level']} | ∅ {row['diameter_m']:.1f}m"
    ).add_to(cluster_layers[cid])

for layer in cluster_layers.values():
    layer.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

output_path = OUTPUT_DIR / "cluster_map.html"
m.save(str(output_path))
print(f"✅ Cluster map saved: {output_path}")

try:
    from IPython.display import display
    display(m)
except Exception:
    pass

### Map Reading

The map confirms and spatialises the clustering results:

- **Cluster 1 (red) — Dense urban**: concentrated in major urban areas (Paris, Lyon, Marseille, Lille), forming dense clusters. These compact roundabouts respond to space constraints in urban centres.

- **Cluster 2 (green) — Peri-urban/rural**: spread across the territory, mainly along national road routes and in peripheral areas. These large gyratories are characteristic of interchanges and entrances to mid-sized towns.

- **Cluster 0 (blue) — Intermediate**: distributed across the whole territory, reflecting transition zones between urban and peri-urban areas.

- **Cluster 3 (orange) — Atypical**: scattered, with no particular geographic concentration. Confirms its interpretation as an outlier group rather than a territorial group.


---

## 🎯 General Conclusion

This project explored the structure of French roundabouts through three complementary data mining approaches.

### What we did

**Part 1 — Collection and analysis:** A complete geospatial processing pipeline was built, from OSM extraction to the construction of an enriched dataset of ~62,000 roundabouts with geometric and territorial attributes.

**Part 2 — Pattern Mining:** The Apriori algorithm (implemented from scratch) extracted association rules revealing clear relationships between roundabout size and territorial context. Small roundabouts are over-represented in dense urban areas; large ones, in peri-urban and rural areas. Certain regions reinforce these trends.

**Part 3 — Clustering:** K-Means (k=4) validated and refined these observations by distinguishing four profiles: dense urban, peri-urban/rural, intermediate, and atypical. The cluster × urbanisation heatmap shows strong convergence with the association rules, while providing additional nuance.

### What we learned

The geography of roundabouts is not random: it reflects land constraints and spatial planning logics. Dense areas favour small, compact gyratories; low-density areas allow more generous infrastructure. These results seem obvious in hindsight — but they were **extracted from data** without prior assumptions, which is precisely the value of data mining.

### Limitations and perspectives

- **OSM data quality**: collaborative data with duplicates and imperfect reconstructions
- **Missing variables**: year of construction, traffic flows, road context — would considerably enrich the analysis
- **Temporal dimension**: OSM does not provide reliable dates — an analysis of historical evolution (via OSM archives) would be an interesting avenue

---
*End of notebook — Perig Montfort*
